# Inferindo
Nesta lição, você irá inferir sentimento e tópicos de avaliações de produtos e artigos de notícias.

## Configuração

In [ ]:
from google import genai
from dotenv import load_dotenv
import os

# Carrega variáveis do .env
load_dotenv()
GOOGLE_API_KEY = os.getenv('API_KEY_IA202601')

client = genai.Client(api_key=GOOGLE_API_KEY)

In [ ]:
from google.genai import types

def get_completion(prompt):
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=0,
            top_p=0.95,
            top_k=20,
        ),
    )
    return response.text

**Nota**: Em junho de 2023, a OpenAI atualizou o gpt-3.5-turbo. Os resultados que você vê no notebook podem ser ligeiramente diferentes dos do vídeo. Alguns dos prompts também foram levemente modificados para produzir os resultados desejados.

## Texto de avaliação de produto

In [ ]:
lamp_review = """
Precisava de um abajur bonito para o meu quarto, e este tinha armazenamento extra e não era muito caro. Chegou rápido. O cordão do abajur quebrou durante o transporte e a empresa enviou um novo rapidamente. Chegou em poucos dias também. Foi fácil de montar. Depois faltou uma peça, então entrei em contato com o suporte e eles rapidamente me enviaram a peça que faltava! Lumina me parece ser uma ótima empresa que se importa com seus clientes e produtos!!
"""

## Sentimento (positivo/negativo)

In [ ]:
prompt = f"""
Qual é o sentimento da seguinte avaliação de produto, que está delimitada por três crases?

Texto da avaliação: '''{lamp_review}'''
"""
response = get_completion(prompt)
print(response)

In [ ]:
prompt = f"""
Qual é o sentimento da seguinte avaliação de produto, que está delimitada por três crases?

Dê sua resposta como uma única palavra, "positivo" ou "negativo".

Texto da avaliação: '''{lamp_review}'''
"""
response = get_completion(prompt)
print(response)

## Identifique tipos de emoções

In [ ]:
prompt = f"""
Identifique uma lista de emoções que o autor da seguinte avaliação está expressando. Inclua no máximo cinco itens na lista. Formate sua resposta como uma lista de palavras minúsculas separadas por vírgulas.

Texto da avaliação: '''{lamp_review}'''
"""
response = get_completion(prompt)
print(response)

## Identifique raiva

In [ ]:
prompt = f"""
O autor da seguinte avaliação está expressando raiva?
A avaliação está delimitada por três crases. Dê sua resposta como sim ou não.

Texto da avaliação: '''{lamp_review}'''
"""
response = get_completion(prompt)
print(response)

## Extraia nome do produto e da empresa de avaliações de clientes

In [ ]:
prompt = f"""
Identifique os seguintes itens do texto da avaliação:
- Item comprado pelo avaliador
- Empresa que fabricou o item

A avaliação está delimitada por três crases. Formate sua resposta como um objeto JSON com as chaves "Item" e "Marca". Se a informação não estiver presente, use "desconhecido" como valor. Seja o mais breve possível.
  
Texto da avaliação: '''{lamp_review}'''
"""
response = get_completion(prompt)
print(response)

## Fazendo múltiplas tarefas ao mesmo tempo

In [ ]:
prompt = f"""
Identifique os seguintes itens do texto da avaliação:
- Sentimento (positivo ou negativo)
- O avaliador está expressando raiva? (true ou false)
- Item comprado pelo avaliador
- Empresa que fabricou o item

A avaliação está delimitada por três crases. Formate sua resposta como um objeto JSON com as chaves "Sentimento", "Raiva", "Item" e "Marca". Se a informação não estiver presente, use "desconhecido" como valor. Seja o mais breve possível. Formate o valor de Raiva como booleano.

Texto da avaliação: '''{lamp_review}'''
"""
response = get_completion(prompt)
print(response)

## Inferindo tópicos

In [ ]:
story = """
Em uma pesquisa recente conduzida pelo governo, funcionários do setor público foram convidados a avaliar seu nível de satisfação com o departamento em que trabalham. Os resultados revelaram que a NASA foi o departamento mais popular, com uma taxa de satisfação de 95%.

Um funcionário da NASA, John Smith, comentou sobre os resultados, dizendo: "Não me surpreende que a NASA tenha ficado em primeiro lugar. É um ótimo lugar para trabalhar, com pessoas incríveis e oportunidades incríveis. Tenho orgulho de fazer parte de uma organização tão inovadora."

Os resultados também foram bem recebidos pela equipe de gestão da NASA, com o diretor Tom Johnson afirmando: "Estamos muito felizes em saber que nossos funcionários estão satisfeitos com seu trabalho na NASA. Temos uma equipe talentosa e dedicada que trabalha incansavelmente para alcançar nossos objetivos, e é fantástico ver que o trabalho duro está valendo a pena."

A pesquisa também revelou que a Administração da Seguridade Social teve a menor taxa de satisfação, com apenas 45% dos funcionários indicando que estavam satisfeitos com seu trabalho. O governo se comprometeu a abordar as preocupações levantadas pelos funcionários na pesquisa e trabalhar para melhorar a satisfação no trabalho em todos os departamentos.
"""

## Inferir 5 tópicos

In [ ]:
prompt = f"""
Determine cinco tópicos que estão sendo discutidos no texto a seguir, que está delimitado por três crases.

Faça cada item com uma ou duas palavras.

Formate sua resposta como uma lista de itens separados por vírgulas.

Exemplo de texto: '''{story}'''
"""
response = get_completion(prompt)
print(response)

In [ ]:
response.split(sep=',')

In [ ]:
topic_list = [
    "nasa", "governo local", "engenharia", 
    "satisfação dos funcionários", "governo federal"
]

## Crie um alerta de notícias para certos tópicos

In [ ]:
prompt = f"""
Determine se cada item da seguinte lista de tópicos é um tópico no texto abaixo, que está delimitado por três crases.

Dê sua resposta assim:
item da lista: 0 ou 1

Lista de tópicos: {", ".join(topic_list)}

Exemplo de texto: '''{story}'''
"""
response = get_completion(prompt)
print(response)

In [ ]:
topic_dict = {i.split(': ')[0]: int(i.split(': ')[1]) for i in response.split(sep='\n')}
if topic_dict['nasa'] == 1:
    print("ALERTA: Nova notícia da NASA!")